In [1]:
import torch
from torch.utils.data import DataLoader
from seqeval.metrics import classification_report
from transformers import AutoTokenizer

from evaluate import evaluate_bert_softmax
from models.bert_softmax import BertSoftmaxNER
from utils.bert.data_utils import read_conll_2,  NERDataset, build_collate_fn

In [2]:
MODEL_NAME = 'bert-base-cased'
MAX_LENGTH = 128
BATCH_SIZE = 16

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

TEST_PATH = '../data/matscholar/test.txt'
MODEL_PATH = '../models/bert_softmax.pt'

In [3]:
# 读取模型文件
checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)

tag2idx = checkpoint['tag2idx']
idx2tag = {v: k for k, v in tag2idx.items()}

# 读取测试数据
test_sentence, test_tags = read_conll_2(TEST_PATH)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

test_dataset = NERDataset(test_sentence, test_tags)

collate_fn = build_collate_fn(
    tokenizer=tokenizer,
    label2id=tag2idx,
    max_length=MAX_LENGTH
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

# 建立模型
model = BertSoftmaxNER(
    model_name=MODEL_NAME,
    num_labels=len(tag2idx),
    id2label=idx2tag,
    label2id=tag2idx,
).to(DEVICE)

model.load_state_dict(checkpoint['model'])

test_loss, test_p, test_r, test_f1, y_true, y_pred = evaluate_bert_softmax(
    model, test_loader, idx2tag, DEVICE
)

print(f"\n========== Test Result ==========")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Precision: {test_p:.4f}")
print(f"Test Recall: {test_r:.4f}")
print(f"Test F1: {test_f1:.4f}")

print(f"\n========== Test Detailed Report ==========")
print(classification_report(y_true, y_pred, digits=4))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



========== Test Result ==========
Test Loss: 0.2351
Test Precision: 0.8021
Test Recall: 0.8418
Test F1: 0.8215

========== Test Detailed Report ==========
              precision    recall  f1-score   support

         APL     0.6429    0.6702    0.6562        94
         CMT     0.8051    0.8533    0.8285       184
         DSC     0.8596    0.8831    0.8712       402
         MAT     0.8502    0.9026    0.8756       698
         PRO     0.7516    0.7697    0.7605       621
         SMT     0.7490    0.8468    0.7949       222
         SPL     0.8293    0.8095    0.8193        42

   micro avg     0.8021    0.8418    0.8215      2263
   macro avg     0.7839    0.8193    0.8009      2263
weighted avg     0.8022    0.8418    0.8213      2263

